<img src="../../../Individual-Contest/Radar/figs/IOAI-Logo.png" alt="IOAI Logo" width="200" height="auto">

[IOAI 2025 (Pekin, Chine), concours individuel](https://ioai-official.org/china-2025)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SKonteye/IOAI-2025/blob/main/fr/Individual-Contest/Radar/Radar.ipynb)

# Radar

## 1. Description du probleme

Le radar est une technologie cle des communications sans fil, largement utilisee par exemple dans les vehicules autonomes. Il repose generalement sur une antenne qui emet des signaux specifiques puis recoit leurs reflexions sur les objets presents dans l'environnement. En traitant ces signaux, le systeme determine la direction angulaire, la distance et la vitesse des objets cibles.

Dans les applications reelles, le traitement des signaux radar est difficile a cause du bruit et des reflexions provenant d'objets non cibles alentour. Par exemple, lorsqu'on cherche a detecter des pietons, le radar peut aussi recevoir des reflexions d'arbres ou d'autres objets d'arriere-plan, ce qui peut degrader la precision. Votre tache consiste a utiliser l'IA pour analyser les signaux recus par le radar et identifier la presence d'un humain a chaque position.

Dans cette tache, nous fournissons un **jeu de donnees d'experience radar en interieur**, et votre objectif est de developper un modele qui realise une **segmentation semantique radar**.

## 2. Jeu de donnees

Pour mesurer les objets autour d'un radar, les parametres cles suivants sont utilises :

- **Distance** : la distance en ligne droite entre le radar et un objet.
- **Azimut** : l'angle horizontal (de gauche a droite) entre le radar et l'objet.
- **Elevation** : l'angle vertical (vers le haut ou vers le bas) de l'objet par rapport au radar.
- **Vitesse** : la vitesse a laquelle l'objet se rapproche ou s'eloigne du radar.

<img src="../../../Individual-Contest/Radar/figs/Radar Fig 1.png" width="300">

Les donnees radar sont traitees sous forme de plusieurs **cartes de chaleur**, chacune encodant l'**intensite du signal recu** a differentes positions et directions.

- Les **cartes de chaleur statiques** mettent l'accent sur les reflexions provenant d'objets **immobiles**.
- Les **cartes de chaleur dynamiques** mettent en evidence les changements causes par des objets **en mouvement**.

Lorsqu'aucun objet n'est present a un endroit donne, le signal est principalement compose de bruit de fond et apparait faible. En revanche, les reflexions d'un objet augmentent l'intensite du signal, ce qui permet sa detection.

Par exemple, la **carte de chaleur statique distance-azimut** represente l'intensite du signal selon differentes distances (**range**) et differents angles horizontaux (**azimuth**), principalement reflechie par des objets immobiles.

Chaque echantillon du jeu de donnees est stocke dans un fichier `.mat.pt` sous forme d'un tenseur de forme $7 \times 50 \times 181$, ou :

- 7 est le nombre de cartes (6 cartes de chaleur + 1 carte d'etiquettes semantiques),
- 50 represente les bins de distance,
- 181 represente les bins angulaires ou de vitesse, couvrant les angles de \-90° a \+90° dans le plan horizontal ou vertical. Vous pouvez supposer que les bins de vitesse sont eux aussi remappes de \-90° a \+90° afin d'assurer une visualisation coherente.
- chaque valeur d'intensite d'une carte de chaleur est normalisee dans [0, 1] et represente l'intensite du signal recu.

Les 6 cartes de chaleur sont organisees comme suit :

- **Indice 0** : carte de chaleur statique distance-azimut
- **Indice 1** : carte de chaleur dynamique distance-azimut
- **Indice 2** : carte de chaleur statique distance-elevation
- **Indice 3** : carte de chaleur dynamique distance-elevation
- **Indice 4** : carte de chaleur statique distance-vitesse
- **Indice 5** : carte de chaleur dynamique distance-vitesse

Toutes les valeurs des cartes de chaleur sont **normalisees** ; aucune conversion d'unites n'est donc necessaire.

La **carte a l'indice 6** est la carte d'etiquettes semantiques, stockee au format distance-azimut.

- **-1** : arriere-plan (aucune cible)
- **0** : valise
- **1** : chaise
- **2** : humain
- **3** : mur

Voici la visualisation de `1.mat.pt` dans `training_set` :

<img src="../../../Individual-Contest/Radar/figs/Radar Fig 2.png" width="675">

Voici une partie d'un echantillon du jeu de donnees :

<img src="../../../Individual-Contest/Radar/figs/Radar Fig 3.png" width="675">

Taille des donnees : 1800 echantillons dans l'ensemble d'entrainement, 500 dans l'ensemble de validation et 500 dans l'ensemble de test.

## 3\. Tache

Votre tache consiste a developper un modele qui prend en entree les **six premieres cartes de chaleur** (indices 0 a 5) et predit en sortie la **carte d'etiquettes semantiques** (indice 6). L'objectif est d'identifier avec precision la nature de la cible (-1 a 3) a chaque position du champ de vision du radar.

1. **Entree** : un tenseur de forme $6 \times 50 \times 181$, representant six cartes de chaleur radar.
2. **Sortie** : un tenseur de forme $50 \times 181$, representant la carte d'etiquettes semantiques des cibles.

## 4\. Soumission

Veuillez soumettre un fichier nomme `submission.ipynb`. La sortie est un fichier zip nomme `submission.zip`, qui contient deux tableaux `submission_val.csv` et `submission_test.csv` correspondant respectivement aux resultats de prediction de l'ensemble de validation et de l'ensemble de test.

**Remarque :** le tableau de sortie doit comporter un en-tete ; les donnees affichees dans le tableau ne sont pas les donnees effectivement resolues, elles servent uniquement d'exemple de format de soumission.

| filename | pixel_0 | pixel_1 | ... | pixel_9049 |
| :------: | :-----: | ------- | --- | ---------- |
| 1.mat.pt |   -1    | -1      | ... | -1         |
|   ...    |   ...   | ...     | ... | ...        |

## 5\. Score

Le score repose sur la **precision de la reconnaissance des etiquettes**. L'identification correcte des points correspondant a des cibles est davantage valorisee que l'identification correcte des points d'arriere-plan.

### Criteres de notation :

* Chaque **pixel d'arriere-plan** correctement identifie rapporte **1 point**.
* Chaque **pixel hors arriere-plan** correctement identifie rapporte **50 points**.
* Le score final est normalise sur **0 a 1** en le comparant au score maximal possible.

### Formule :
$$
Score = \frac{|C_{0,correct}| \times 1 + |C_{1,correct}| \times bonus}{|C_0| \times 1 + |C_1| \times bonus}
$$
ou :
$$
\begin{aligned}
I &= \{1, 2, \dots, 50\times 181\}\\
C_0 &= \{i \in I \mid y_i = -1\}\\
C_1 &= \{i \in I \mid y_i \neq -1\}\\
C_{0,correct} &= \{i \in C_0 \mid p_i = y_i\}\\
C_{1,correct} &= \{i \in C_1 \mid p_i = y_i\}\\
\end{aligned}
$$

### Exemple

Pour une carte de chaleur $3\times3$, supposons que la verite terrain soit :
$$
\begin{bmatrix}
-1 & -1 & -1 \\
1 & 2 & 3 \\
-1 & -1 & -1
\end{bmatrix}
$$

Le resultat prevu est :
$$
\begin{bmatrix}
-1 & 1 & -1 \\
-1 & 2 & -1 \\
-1 & 3 & -1
\end{bmatrix}
$$

Il y a alors quatre `-1` correctement identifies et un `2` correctement identifie. Votre score est 4 + 50 = 54 points. Le score maximal possible est 6 + 50 * 3 = 156, c'est-a-dire le score pour six pixels d'arriere-plan et trois pixels hors arriere-plan. Votre score normalise est 54 / 156 = 0.346.
$$
Score = \frac{4 \times 1 + 1 \times 50}{6 \times 1 + 3 \times 50}=0.346
$$

## 6. Baseline et ensemble d'entrainement

- Vous trouverez ci-dessous la solution de reference de base.
- Le jeu de donnees se trouve dans le dossier `training_set`.
- Le meilleur score obtenu par le Comite scientifique pour cette tache est de 0.90 dans le leaderboard B ; ce score est utilise pour l'unification des scores.
- Le score de base obtenu par le Comite scientifique pour cette tache est de 0.67 dans le leaderboard B ; ce score est utilise pour l'unification des scores.

### Chargement des donnees

In [ ]:
import random
import numpy as np
import torch

seed = 42

random.seed(seed)                  # generateur aleatoire integre de Python
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (un seul GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (tous les GPU)

# Garantit un comportement deterministe
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class CustomDataset(Dataset):
    def __init__(self, file_paths, transform=None):
        self.file_paths = file_paths
        self.transform = transform
        self.file_names = [os.path.basename(path) for path in file_paths]

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        data = torch.load(self.file_paths[idx], weights_only=True)
        
        images = data[:6]  
        labels = data[6]           
        
        images = images.float()  
        labels = labels.long()   
        labels = labels + 1

        if self.transform:
            images = self.transform(images)
            labels = self.transform(labels)
        
        return images, labels, self.file_names[idx]

class CustomDataset_test(Dataset):
    def __init__(self, file_paths, transform=None):
        self.file_paths = file_paths
        self.transform = transform
        self.file_names = [os.path.basename(path) for path in file_paths]

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        data = torch.load(self.file_paths[idx], weights_only=True)
        
        images = data[:6]        
        
        images = images.float()  

        if self.transform:
            images = self.transform(images)
        
        return images, self.file_names[idx]

def generate_file_paths(base_path):
    file_paths = []
    for frame in os.listdir(base_path):
        frame_path = os.path.join(base_path, frame)
        if frame_path.endswith('.mat.pt'):
            file_paths.append(frame_path)
    return [path for path in file_paths if os.path.exists(path)]

def load_data(base_path, batch_size=4, num_workers=2, test_size=0.2):
    file_paths = generate_file_paths(base_path)
    
    train_paths, test_paths = train_test_split(file_paths, test_size=test_size, random_state=42)
    
    train_dataset = CustomDataset(file_paths=train_paths)
    test_dataset = CustomDataset(file_paths=test_paths)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=num_workers, 
        drop_last=True
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers, 
        drop_last=True
    )
    
    return train_loader, test_loader

### Definition du modele et entrainement

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=3, padding=1)  
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)  
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=5, kernel_size=3, padding=1)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.conv3(x)  
        return x

def train(model, train_loader, test_loader, optimizer, criterion, num_epochs=100):
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        batch_count = 0
        
        for images, labels, _ in train_loader:
            images = images.cuda() if torch.cuda.is_available() else images
            labels = labels.cuda() if torch.cuda.is_available() else labels
            
            outputs = model(images)
            outputs = outputs.view(outputs.size(0), outputs.size(1), -1)  # [B, C, H*W]
            labels = labels.view(labels.size(0), -1)  # [B, H*W]
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            batch_count += 1
        
        avg_train_loss = epoch_loss / batch_count
        train_losses.append(avg_train_loss)
        
        model.eval()
        val_loss = 0.0
        val_batch_count = 0
        
        with torch.no_grad():
            for images, labels, _ in test_loader:
                images = images.cuda() if torch.cuda.is_available() else images
                labels = labels.cuda() if torch.cuda.is_available() else labels
                
                outputs = model(images)
                outputs = outputs.view(outputs.size(0), outputs.size(1), -1)
                labels = labels.view(labels.size(0), -1)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                val_batch_count += 1
        
        avg_val_loss = val_loss / val_batch_count
        val_losses.append(avg_val_loss)
        
        if (epoch+1) % 2 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], ' \n                  f'Train Loss: {avg_train_loss:.4f}, ' \n                  f'Val Loss: {avg_val_loss:.4f}')
    
    return train_losses, val_losses
TRAIN_PATH = "./"
# L'ensemble d'entrainement est deploye automatiquement sur la machine de test.
# Votre notebook peut acceder a TRAIN_PATH meme si vous ne le montez pas avec le notebook.
data_path = TRAIN_PATH + 'training_set'

train_loader, test_loader = load_data(
    base_path=data_path,
    batch_size=4,  
    num_workers=2,
    test_size=0.2
)

model = MyModel()
if torch.cuda.is_available():
    model = model.cuda()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001) 

train_losses, val_losses = train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    num_epochs=40
)

### Generer les CSV de soumission

In [ ]:
# Executer l'inference sur l'ensemble de validation et l'ensemble de test
from torch.utils.data import DataLoader
import pandas as pd

def run_inference(model, data_loader):
    """Executer l'inference et renvoyer les predictions avec les noms de fichiers"""
    model.eval()
    predictions = []
    filenames = []
    
    with torch.no_grad():
        for images, file_names in data_loader:
            images = images.cuda() if torch.cuda.is_available() else images
            
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            
            # Reconvertir les predictions vers l'intervalle d'etiquettes original [-1, 3]
            preds = preds - 1
            
            # Aplatir les predictions pour chaque echantillon
            for i, pred in enumerate(preds):
                predictions.append(pred.cpu().numpy().flatten())
                filenames.append(file_names[i])
    
    return predictions, filenames

# DATA_PATH est la variable d'environnement secrete qui indique l'adresse de l'ensemble de validation et de l'ensemble de test sur la machine de test.
# Vous ne pouvez pas acceder a cette adresse localement.
if os.environ.get('DATA_PATH'):
    DATA_PATH = os.environ.get("DATA_PATH") + "/" 
else:
    DATA_PATH = "Solution/"  # Repli pour les tests locaux
# Charger l'ensemble de validation
val_paths = generate_file_paths(DATA_PATH + 'validation_set')
val_dataset = CustomDataset_test(file_paths=val_paths)
val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2
)

# Charger l'ensemble de test
test_paths = generate_file_paths(DATA_PATH + 'test_set')
test_dataset = CustomDataset_test(file_paths=test_paths)
test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2
)

# Executer l'inference sur l'ensemble de validation
print("Running inference on validation set...")
val_predictions, val_filenames = run_inference(model, val_loader)

# Enregistrer les resultats de validation dans un CSV
val_results = []
for filename, pred in zip(val_filenames, val_predictions):
    # Creer une ligne avec le nom de fichier et les predictions aplaties
    row = {'filename': filename}
    for i, p in enumerate(pred):
        row[f'pixel_{i}'] = p
    val_results.append(row)

val_df = pd.DataFrame(val_results)
val_df.to_csv('submission_val.csv', index=False)
print(f"Validation results saved to output_validation.csv with shape: {val_df.shape}")

# Executer l'inference sur l'ensemble de test
print("Running inference on testing set...")
test_predictions, test_filenames = run_inference(model, test_loader)

# Enregistrer les resultats de test dans un CSV
test_results = []
for filename, pred in zip(test_filenames, test_predictions):
    # Creer une ligne avec le nom de fichier et les predictions aplaties
    row = {'filename': filename}
    for i, p in enumerate(pred):
        row[f'pixel_{i}'] = p
    test_results.append(row)

test_df = pd.DataFrame(test_results)
test_df.to_csv('submission_test.csv', index=False)
print(f"Testing results saved to output_testing.csv with shape: {test_df.shape}")

print("\nInference completed! Results saved to:")
print("- submission_val.csv (for validation set leaderboard)")
print("- submission_test.csv (for testing set leaderboard)")

### Creer le fichier .zip

In [ ]:
import zipfile
import os

# Definir les fichiers a compresser et le nom du fichier zip.
files_to_zip = ['submission_val.csv', 'submission_test.csv']
zip_filename = 'submission.zip'

# Creer un fichier zip
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        # Ajouter le fichier a l'archive zip
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} Created successfully!')